Hemos decidido limpiar
- Nombre del pasajero: no aporta ninguna información relevante y no vamos a numerizarlo

In [5]:
import re
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Cargar datos
df = pd.read_csv('dataset.csv')

# Limpieza básica del texto
def clean_name(name):
    if pd.isna(name):
        return 'Desconocido'
    name = str(name).strip()
    name = re.sub(r'\s+', ' ', name)
    return name

df['Name'] = df['Name'].apply(clean_name)
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).fillna('Desconocido').str.strip()
df['Surname'] = df['Name'].str.split(',').str[0].fillna('Desconocido').str.strip()

# Reemplazo explícito de nulos en Cabin
df['Cabin'] = df['Cabin'].fillna('Desconocido').astype(str).str.strip()

# Otras columnas con valores faltantes
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode(dropna=True)[0])
df['Fare'] = pd.to_numeric(df['Fare'], errors='coerce')
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Variables derivadas útiles
df['FamilySize'] = df['SibSp'].fillna(0) + df['Parch'].fillna(0) + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Limpieza final del nombre: se elimina porque no aporta señal directa al modelo
df = df.drop(columns=['Name'])

# Separar objetivo si existe
y = df['Survived'] if 'Survived' in df.columns else None
X = df.drop(columns=['Survived']) if 'Survived' in df.columns else df.copy()

# Columnas numéricas y categóricas
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Sex', 'Ticket', 'Cabin', 'Embarked', 'Title', 'Surname']

numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

X_preprocessed = preprocessor.fit_transform(X)

# Guardar resultados para reutilizarlos después
np.save('X_preprocessed.npy', X_preprocessed)
joblib.dump(preprocessor, 'preprocessor.joblib')

print('Datos originales:', df.shape)
print('Datos preprocesados:', X_preprocessed.shape)
print(df.head())

Datos originales: (889, 15)
Datos preprocesados: (889, 1521)
   PassengerId  Survived  Pclass     Sex   Age  SibSp  Parch  \
0            1         0       3    male  22.0      1      0   
1            2         1       1  female  38.0      1      0   
2            3         1       3  female  26.0      0      0   
3            4         1       1  female  35.0      1      0   
4            5         0       3    male  35.0      0      0   

             Ticket     Fare        Cabin Embarked Title    Surname  \
0         A/5 21171   7.2500  Desconocido        S    Mr     Braund   
1          PC 17599  71.2833          C85        C   Mrs    Cumings   
2  STON/O2. 3101282   7.9250  Desconocido        S  Miss  Heikkinen   
3            113803  53.1000         C123        S   Mrs   Futrelle   
4            373450   8.0500  Desconocido        S    Mr      Allen   

   FamilySize  IsAlone  
0           2        0  
1           2        0  
2           1        1  
3           2        0  
4 